In [ ]:

# -- Cell 1 -- rclone + Drive. Same pattern as every other notebook here.
# Requires: Settings -> Internet ON, Accelerator GPU T4 x2, RCLONE_DRIVE_TOKEN.
import os, subprocess

r = subprocess.run("curl -s https://rclone.org/install.sh | sudo bash", shell=True)
if r.returncode not in (0, 3):
    raise RuntimeError("rclone install failed (exit %d)" % r.returncode)

from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("RCLONE_DRIVE_TOKEN")
os.makedirs("/root/.config/rclone", exist_ok=True)
with open("/root/.config/rclone/rclone.conf", "w") as f:
    f.write("[drive]\ntype = drive\nscope = drive\ntoken = " + token + "\n")

REMOTE = "drive:Distillation"
out = subprocess.run("rclone lsf " + REMOTE, shell=True, capture_output=True, text=True)
print(out.stdout or out.stderr)
assert out.returncode == 0, "cannot see " + REMOTE


In [ ]:

# -- Cell 2 -- deps + GPU.
#
# THIS NOTEBOOK IS THE ANALYSIS PASS, not the benchmark pass. It decides WHICH
# compressed models are worth evaluating, and exports them. The expensive
# finetuning runs live in the follow-up notebook, because they should not start
# until the depth curve says there is anything to evaluate.
#
# Nothing here trains. Every measurement is a forward pass plus a linear probe.
subprocess.run('pip install -q -U "transformers>=5.0"', shell=True, check=True)
subprocess.run("pip uninstall -y -q torchao", shell=True)

import torch, numpy as np, pandas as pd, glob, json, time
print("torch", torch.__version__, "| GPUs", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print("   cuda:%d %s %.0f GB" % (i, p.name, p.total_memory / 1e9))
assert torch.cuda.device_count() >= 1, "need a GPU"


In [ ]:

# -- Cell 3 -- their data, our code, both model scales.
#
# Only their_repo/data and their_repo/training come down: 401 of the repo's 468
# files are figure_generation/, which nothing here reads, and Drive charges per
# file.
WORK = "/kaggle/working"
CODE, REPO = WORK + "/distill", WORK + "/project/their_repo"
TEACH = WORK + "/models/peptideclm-2-mlm-large"
SMALL = WORK + "/models/peptideclm-2-mlm-small"
os.makedirs(REPO, exist_ok=True)

def rlsf(path):
    r = subprocess.run("rclone lsf " + path, shell=True, capture_output=True, text=True)
    return r.stdout.split() if r.returncode == 0 else []

def pull_model(name, local):
    """Drive holds the 32M flat and the 337M in HF cache layout, so try both and
    name the paths tried rather than raising IndexError on an empty listing."""
    if os.path.exists(local + "/model.safetensors"):
        return "already local"
    flat = "%s/models/%s" % (REMOTE, name)
    if any(f.startswith("model.safetensors") for f in rlsf(flat)):
        os.makedirs(local, exist_ok=True)
        subprocess.run("rclone copy %s %s -P" % (flat, local), shell=True, check=True)
        return "flat"
    snaps = "%s/models/models--aaronfeller--%s/snapshots" % (REMOTE, name)
    shas = [x.rstrip("/") for x in rlsf(snaps)]
    assert shas, "%s not found. Tried  %s  and  %s" % (name, flat, snaps)
    os.makedirs(local, exist_ok=True)
    subprocess.run("rclone copy %s/%s %s -P" % (snaps, shas[0], local),
                   shell=True, check=True)
    return "snapshot " + shas[0][:12]

for sub in ("data", "training"):
    if not os.path.isdir(REPO + "/" + sub):
        subprocess.run("rclone copy %s/their_repo/%s %s/%s --transfers 16 -P"
                       % (REMOTE, sub, REPO, sub), shell=True, check=True)
if not os.path.exists(CODE + "/probe_depth.py"):
    subprocess.run("rclone copy %s/distill %s --transfers 8 -P" % (REMOTE, CODE),
                   shell=True, check=True)

print("teacher <- %s" % pull_model("peptideclm-2-mlm-large", TEACH))
print("small   <- %s" % pull_model("peptideclm-2-mlm-small", SMALL))

need = ["probe_depth.py", "probe_bi.py", "export_truncated.py", "bench_control.py",
        "student.py"]
have = sorted(os.path.basename(f) for f in glob.glob(CODE + "/*.py"))
missing = [f for f in need if f not in have]
assert not missing, "upload to drive:Distillation/distill -- %s" % missing
print("code:", have)

# The scripts resolve data and models relative to a project root, so mirror the
# layout they expect instead of passing paths through every call site.
os.makedirs(WORK + "/models", exist_ok=True)
if not os.path.islink(WORK + "/their_repo") and not os.path.exists(WORK + "/their_repo"):
    os.symlink(REPO, WORK + "/their_repo")
for v, dst in (("large", "models--aaronfeller--peptideclm-2-mlm-large"),
               ("small", "models--aaronfeller--peptideclm-2-mlm-small")):
    d = "%s/models/%s/snapshots/local" % (WORK, dst)
    if not os.path.exists(d):
        os.makedirs(os.path.dirname(d), exist_ok=True)
        os.symlink(TEACH if v == "large" else SMALL, d)
print("layout ready under", WORK)


In [ ]:

# -- Cell 4 -- benchmark controls, and the THPep split their repo does not ship.
#
# WHY THIS RUNS BEFORE ANY GPU WORK. A benchmark that a 405-dim token histogram
# solves cannot tell a compressed encoder from an uncompressed one, so measuring
# compression on it is wasted GPU time. Locally this already eliminated CellPPD:
# bag-of-tokens 0.8270 against the full 337M encoder's 0.8278.
#
# It also writes THPep_train.csv / THPep_test.csv. Their classification script
# looks for those and crashes without them. The split is OURS -- fine for
# comparing our arms, NOT comparable to the paper's THPep numbers.
r = subprocess.run(["python", "bench_control.py"], cwd=CODE,
                   capture_output=True, text=True)
print(r.stdout[-3000:])
if r.returncode != 0:
    print("----- STDERR -----"); print(r.stderr[-2500:])
assert r.returncode == 0, "control failed"


In [ ]:

# -- Cell 5 -- depth curve on AmpHGT. THE DECISION CELL.
#
# AmpHGT is the instrument: 9,265 train (so n > d = 1024 and the probe is
# well-posed) and 5,148 test, which is 17x CellPPD's test set and shrinks the
# confidence interval about 4x. On CellPPD the bootstrap CI came out WIDER than
# the entire spread across depths, so no depth could be chosen from it.
#
# This measures two things at once:
#   1. the encoder's own linear-probe score, which is the number the bag-of-tokens
#      control (0.6853 on AmpHGT) has to be compared against
#   2. MCC vs depth, which bounds every depth-pruning method -- no smarter
#      criterion beats cutting where the representation saturates
#
# Embeddings are cached to results/, so re-analysis afterwards is seconds.
#
# REUSE A PREVIOUS RUN'S CACHE IF ONE IS ATTACHED. Extraction is ~55 min of GPU;
# everything after it is CPU. Attach the earlier notebook via Add Input ->
# Notebook Output, and its results/ mounts under /kaggle/input/<name>/. Copying
# the .npz across skips the extraction entirely.
# Two ways to supply it, whichever is easier:
#   a) Add Input -> Notebook Output -> the earlier notebook  (mounts /kaggle/input/)
#   b) upload the .npz to drive:Distillation/results/        (pulled here)
# Note Kaggle serves the file as .zip on download -- an .npz IS a zip archive, so
# renaming the extension back is all that is needed before uploading to Drive.
os.makedirs(WORK + "/results", exist_ok=True)
subprocess.run("rclone copy %s/results %s/results --include 'depth_emb_*_AmpHGT.npz' -P"
               % (REMOTE, WORK), shell=True, check=False)
found = (glob.glob("/kaggle/input/*/results/depth_emb_*_AmpHGT.npz")
         + glob.glob(WORK + "/results/depth_emb_*_AmpHGT.npz"))
for f in found:
    dst = WORK + "/results/" + os.path.basename(f)
    if not os.path.exists(dst):
        subprocess.run(["cp", f, dst], check=True)
    print("reusing cached embeddings: %s (%.2f GB)" % (f, os.path.getsize(f) / 1e9))
if not found:
    print("no cached embeddings attached -- extracting from scratch (~55 min GPU)")

t0 = time.time()
r = subprocess.run(["python", "-u", "probe_depth.py", "--bench", "AmpHGT",
                    "--batch", "32", "--fast"],
                   cwd=CODE, capture_output=True, text=True)
print(r.stdout[-6000:])
if r.returncode != 0:
    print("----- STDERR -----"); print(r.stderr[-3000:])
assert r.returncode == 0, "depth probe failed"
print("\ndepth probe took %.1f min" % ((time.time() - t0) / 60))


In [ ]:

# -- Cell 6 -- read the curve and pick the truncation depth.
#
# The rule is deliberately conservative: the SHALLOWEST depth whose 95% bootstrap
# CI still reaches the full stack's point estimate. Taking the argmax over 33
# noisy depths would name a winner even if the true curve were flat -- that is
# exactly what happened on CellPPD, where the first run reported "depth 14, +0.046"
# and the number was noise.
df = pd.read_csv(glob.glob("/kaggle/working/results/depth_probe_*AmpHGT.csv")[0]
                 if glob.glob("/kaggle/working/results/depth_probe_*AmpHGT.csv")
                 else glob.glob(CODE + "/../results/depth_probe_*AmpHGT.csv")[0])
full = df.iloc[-1]
cand = df[df.mcc_hi >= full.mcc]
DEPTH = int(cand.iloc[0].depth) if len(cand) else int(full.depth)

print(df[["depth", "params_M", "mcc", "mcc_lo", "mcc_hi", "auc"]].to_string(index=False))
print("\nfull depth %d: MCC %.4f [%.4f, %.4f]" % (full.depth, full.mcc, full.mcc_lo, full.mcc_hi))
print("chosen truncation depth: %d  (%.1fM, %.0f%% of the teacher)"
      % (DEPTH, df[df.depth == DEPTH].params_M.iloc[0],
         100 * df[df.depth == DEPTH].params_M.iloc[0] / full.params_M))
print("bag-of-tokens control on AmpHGT was 0.6853 -- the encoder must clear that")
print("for this benchmark to be measuring the encoder at all.")


In [ ]:

# -- Cell 7 -- Block Influence, to find dead blocks in the MIDDLE.
#
# Truncation removes a suffix. BI = 1 - E[cos(block input, block output)] finds
# pass-through blocks anywhere in the stack.
#
# Read the output carefully rather than trusting the ranking. Locally the profile
# came out as: block 0 = 0.76, block 31 = 0.60, and every block between them
# 0.002-0.022. In a pre-LN residual network each block adds a small delta to a
# large and growing residual stream, so cosine-based influence is compressed
# toward zero for depth -- which means BI here separates the endpoints from the
# middle but barely discriminates WITHIN the middle. Treat the ordering among
# near-ties as arbitrary, and let the exported-and-evaluated result decide.
r = subprocess.run(["python", "-u", "probe_bi.py", "--calib", "AmpHGT,PAMPA,THPep",
                    "--n-per", "400", "--batch", "16"],
                   cwd=CODE, capture_output=True, text=True)
print(r.stdout[-5000:])
if r.returncode != 0:
    print("----- STDERR -----"); print(r.stderr[-2500:])
assert r.returncode == 0, "BI scoring failed"


In [ ]:

# -- Cell 8 -- export the candidates, verify each, ship to Drive.
#
# Each export is checked against the original model with the same blocks bypassed
# at runtime; they must agree to floating-point noise. That catches the one bug
# that matters here -- state-dict keys are blocks.<i>.*, and nn.ModuleList indexes
# positionally, so survivors have to be renumbered. Miss it and blocks load
# randomly initialised, which reads as "pruning hurt a lot" rather than as a bug.
DEST = REMOTE + "/results/compress_analysis"

# MEASUREMENTS GO TO DRIVE FIRST, before anything that can fail.
# The previous version exported first and uploaded last, so one failing variant
# killed the cell and an hour of probing reached Drive as nothing at all. In batch
# mode there is no session left to recover it from.
subprocess.run("rclone copy %s/results %s/analysis --exclude '*.npz' "
               "--drive-chunk-size 64M -P" % (WORK, DEST), shell=True, check=False)
print("analysis results safe on Drive")

bi = json.load(open(WORK + "/results/block_influence.json"))
order = bi["removal_order"]
EXPORT = WORK + "/compressed"
os.makedirs(EXPORT, exist_ok=True)

# DEPTHS ARE CHOSEN FROM THE SHAPE OF THE CURVE, NOT BY ARGMAX OR BY THE RULE.
# The AmpHGT curve came out W-shaped, not flat and not monotonic: emb 0.7208,
# d1 0.5643, d3 0.7445, d6 0.7630, d8-d26 mostly BELOW the bag-of-tokens control,
# d31 0.7681, d32 0.7605. Adjacent depths differ by up to 10x the CI width, so the
# oscillation is real. On a curve like that "shallowest depth whose CI reaches the
# full stack" lands on a spike with a collapse on either side, which is why the
# first run proposed depth 3 alone.
#
#   trunc3   32.3M  size-matched to the released 32M (31.7M) -- the sharp question
#                   is whether the teacher's first 3 blocks beat a model trained
#                   from scratch at that size
#   trunc6   63.8M  best shallow point, and its neighbours (5, 7) are also >=0.754,
#                   so it is not a lone spike
#   trunc31 326.2M  global max; removing only the last block IMPROVES the probe,
#                   consistent with that block being the most MLM-specialised
DEPTH_SET = sorted({3, 6, 31, DEPTH})
variants = [("trunc%d" % d, ["--depth", str(d)]) for d in DEPTH_SET]
for k in (4, 8):
    drop = sorted(order[:k])
    variants.append(("bi_drop%d" % k, ["--drop", ",".join(map(str, drop))]))

# One variant failing must not cost the others. Collect outcomes, report at the
# end, and only assert once everything that CAN be produced has been.
made, failed = [], []
for name, args in variants:
    d = "%s/peptideclm-2-mlm-%s" % (EXPORT, name)
    r = subprocess.run(["python", "export_truncated.py", "--out", d] + args,
                       cwd=CODE, capture_output=True, text=True)
    print("== %s ==" % name); print(r.stdout[-900:])
    if r.returncode == 0:
        made.append(d)
        subprocess.run("rclone copy %s %s/models/%s --drive-chunk-size 64M -P"
                       % (d, DEST, os.path.basename(d)), shell=True, check=False)
    else:
        print(r.stderr[-1200:])
        failed.append(name)

print("")
print("exported: %s" % [os.path.basename(d) for d in made])
if failed:
    print("FAILED:   %s  -- investigate before benchmarking these" % failed)
print("\nexported and uploaded:", [os.path.basename(d) for d in made])
print("-> " + DEST)
print("\nNext notebook runs the benchmarks on these against the 32M warm-start")
print("and the full 337M teacher. Do not start it if the depth curve above was flat")
print("at the bottom -- that would mean there is no removable depth to evaluate.")


In [ ]:

# -- Cell 9 -- is the depth curve's shape real, or the final LayerNorm?
#
# Every depth above was pooled AFTER the stack's final LayerNorm, because that is
# what a truncated model computes. But that norm was trained for block-32 outputs,
# and its per-token rescaling is not something a linear probe can undo. So part of
# the W shape could be norm mismatch rather than representation quality.
#
# The embedding cache from Cell 5 holds BOTH variants, so this is CPU-only: no
# extraction, no GPU. A few depths are enough -- the collapse (1), the shallow
# spike (3), the best shallow point (6), the trough (18), and the top (31, 32).
#
# Read it as: large, depth-VARYING delta -> the norm is distorting the comparison
# and the shallow depths need re-checking. Small or uniform delta -> the shape is
# in the representation and the candidates stand.
r = subprocess.run(["python", "-u", "probe_depth.py", "--bench", "AmpHGT",
                    "--depths", "1,3,6,18,31,32"],
                   cwd=CODE, capture_output=True, text=True)
print(r.stdout[-3000:])
if r.returncode != 0:
    print("----- STDERR -----"); print(r.stderr[-2000:])
subprocess.run("rclone copy %s/results %s/analysis --exclude '*.npz' "
               "--drive-chunk-size 64M -P" % (WORK, DEST), shell=True, check=False)
